# Lab 39: Model Evaluation, Overfitting and Hyperparameter Tuning

**Duration:** 3 hours. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A number such as *97 % accuracy* means little until you know **how it was measured**. In this lab you learn to evaluate a model like an engineer. You draw **learning curves** to see if more data would help, **validation curves** to find the right size of a model, use **cross-validation** to see how much a score jumps around, tune **hyperparameters** one at a time and then with a grid search, and see with your own eyes how tuning on the test set gives a score that is too good to be true.

**By the end of this lab you can:**
1. Draw and read a learning curve and say if the model needs more data or a different model.
2. Draw a validation curve for one hyperparameter and pick its value without using the test set.
3. Explain the variance of a score and use cross-validation to measure it.
4. Tune hyperparameters one at a time and with a grid search, and keep the test set for one final exam.
5. Show that tuning on the test set gives an optimistic score, and explain why.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Evaluation and tuning* of classical models, with scikit-learn on the handwritten digits (flattened to 64 numbers). The ideas are the same for any model, also the deep networks of the next labs.

## 2. Background

**Parameters and hyperparameters.** **Parameters** are the numbers the model learns (weights, the questions of a tree). **Hyperparameters** are the settings that *you* choose before the training: the depth of a tree, `k` of the nearest neighbours, the learning rate, the number of epochs. Tuning means finding good hyperparameters.

| Model | Hyperparameter | Too small | Too big |
|---|---|---|---|
| Decision tree | `max_depth` | Underfits (too simple) | Overfits (memorises) |
| k nearest neighbours | `n_neighbors` (k) | Overfits (noisy) | Underfits (too smooth) |
| Random forest | `n_estimators` (trees) | Unstable | Slower, little gain |

**Underfitting and overfitting.**

| | Training score | Validation score | What to do |
|---|---|---|---|
| **Underfitting** (high bias) | Low | Low | A bigger or better model, more features |
| **Overfitting** (high variance) | High | Much lower | More data, a simpler model, stop earlier |
| Good fit | High | Close to it | Done |

**Learning curve.** The score of the training set and of the validation set as the **number of training examples** grows. If both curves are low and close, more data will not help: the model is too simple. If there is a big gap, more data (or a simpler model) helps.

**Validation curve.** The same two scores as one **hyperparameter** grows. The best value is where the validation score is highest. Before it, the model underfits. After it, the validation score either falls (overfitting) or stays flat (more complexity no longer helps) while the training score keeps rising.

**Variance of a score.** The score changes with the random split. Cross-validation repeats the split and reports the **mean and the spread**. If two models differ by less than the spread, you cannot say which one is better.

**Rules for tuning.**
1. Change **one** hyperparameter at a time, and write down the result.
2. Choose with the **validation** score (or cross-validation), never with the test set.
3. Use the **test set once**, at the end, for the model you chose.
4. If you used the test set to choose, it is a validation set. Then you have no test set left.

**Grid search.** Try every combination of a few values for two or three hyperparameters, with cross-validation for each one. `GridSearchCV` does it for you. It grows fast: 5 values x 4 values x 5 folds = 100 trainings.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(39)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, learning_curve, train_test_split, validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

SEED = 5
digits = load_digits()
X, y = digits.data / 16.0, digits.target
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)       # the test set: not used until Part F
print("development set (train + validation):", len(X_rest), "| test set (locked):", len(X_test), "| features:", X.shape[1])

---
# Part B: One Score Is Not Enough (30 min)

### Step 1: How Much Does the Score Jump?
The cell trains the same decision tree on 30 different random splits of the development set and prints how the test score on the validation part changes. Then it does the same with 5-fold cross-validation.

In [ ]:
scores = []
for seed in range(30):
    a, b, ya, yb = train_test_split(X_rest, y_rest, test_size=0.25, random_state=seed)
    scores.append(DecisionTreeClassifier(max_depth=8, random_state=SEED).fit(a, ya).score(b, yb))
scores = np.array(scores)
print(f"30 random splits: lowest {scores.min() * 100:.1f} %  highest {scores.max() * 100:.1f} %  mean {scores.mean() * 100:.1f} %  spread (std) {scores.std() * 100:.1f} %")
cv = cross_val_score(DecisionTreeClassifier(max_depth=8, random_state=SEED), X_rest, y_rest, cv=5)
print(f"5-fold cross-validation: {[round(float(s), 3) for s in cv]}  mean {cv.mean() * 100:.1f} %  spread {cv.std() * 100:.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The lowest and the highest score of the 30 splits:
- The mean and the spread of the cross-validation:
- If two models differ by 0.5 % and the spread is 1.5 %, can you say which one is better:

---
# Part C: Learning Curves (35 min)

### Step 1: Does More Data Help?
`learning_curve` trains the model with 10 %, 20 % ... 100 % of the training data and gives the cross-validated scores. The cell draws two models: a tree that can grow without limit and a tree with depth 3.

In [ ]:
sizes = np.linspace(0.1, 1.0, 8)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (title, model) in zip(axes, (("tree, no depth limit", DecisionTreeClassifier(random_state=SEED)), ("tree, depth 3", DecisionTreeClassifier(max_depth=3, random_state=SEED)))):
    n_used, train_scores, val_scores = learning_curve(model, X_rest, y_rest, train_sizes=sizes, cv=5, random_state=SEED)
    ax.plot(n_used, train_scores.mean(axis=1), "o-", label="train")
    ax.plot(n_used, val_scores.mean(axis=1), "o-", label="validation")
    ax.set_title(title)
    ax.set_xlabel("training examples")
    ax.legend()
    ax.grid(alpha=0.3)
    print(f"{title:<22} at the end: train {train_scores.mean(axis=1)[-1] * 100:.0f} %  validation {val_scores.mean(axis=1)[-1] * 100:.0f} %")
axes[0].set_ylabel("accuracy")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- For which tree is there a big gap between the two curves, and what is the problem:
- For which tree are both curves low, and what is the problem:
- Would 10 times more data help the depth-3 tree? And the tree without limit:

---
# Part D: Validation Curves (35 min)

### Step 1: The Depth of the Tree
`validation_curve` trains the model for each value of one hyperparameter. The best depth is where the validation line peaks.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, 20]
train_scores, val_scores = validation_curve(DecisionTreeClassifier(random_state=SEED), X_rest, y_rest, param_name="max_depth", param_range=depths, cv=5)
plt.figure(figsize=(5.5, 3.4))
plt.plot(depths, train_scores.mean(axis=1), "o-", label="train")
plt.plot(depths, val_scores.mean(axis=1), "o-", label="validation")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
best = depths[int(val_scores.mean(axis=1).argmax())]
print("Best depth by the validation score:", best, "| validation accuracy", round(float(val_scores.mean(axis=1).max()), 3))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The best depth and its validation accuracy:
- What happens to the train score and to the validation score at large depths (does the validation score fall, or does it stay flat):
- Where does the model underfit (low depths) and where is more depth no longer useful:

### Step 2: The Same for k of the Nearest Neighbours
Change only `n_neighbors`. Which side of the curve is overfitting here: a small k or a big k?

In [ ]:
ks = [1, 3, 5, 9, 15, 25, 45]
train_scores, val_scores = validation_curve(KNeighborsClassifier(), X_rest, y_rest, param_name="n_neighbors", param_range=ks, cv=5)
for k, tr, va in zip(ks, train_scores.mean(axis=1), val_scores.mean(axis=1)):
    print(f"k = {k:>2}: train {tr * 100:5.1f} %  validation {va * 100:5.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The best k:
- The train and validation scores for k = 1:
- Which side overfits: a small k or a big k, and why:

---
# Part E: Grid Search (40 min)

### Step 1: Two Hyperparameters of a Random Forest
`GridSearchCV` tries every combination with 5-fold cross-validation and remembers the best. The grid has 3 x 4 = 12 combinations, so 60 trainings. The cell prints the time and a table of the results.

In [ ]:
grid = {"n_estimators": [10, 50, 150], "max_depth": [3, 6, 12, None]}
started = time.time()
search = GridSearchCV(RandomForestClassifier(random_state=SEED), grid, cv=5, n_jobs=-1).fit(X_rest, y_rest)
print(f"{len(search.cv_results_['params'])} combinations x 5 folds = {len(search.cv_results_['params']) * 5} trainings in {time.time() - started:.0f} s")
table = pd.DataFrame(search.cv_results_["params"]).assign(mean=search.cv_results_["mean_test_score"].round(3), spread=search.cv_results_["std_test_score"].round(3))
print(table.sort_values("mean", ascending=False).head(6).to_string(index=False))
print("Best:", search.best_params_, "| cross-validated accuracy", round(search.best_score_, 3))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The best combination and its score:
- Is the best combination clearly better than the second best? Compare the difference with the spread:
- How many trainings would a grid of 5 x 5 x 5 values and 5 folds need:

---
# Part F: The Honest Exam, and the Trap (35 min)

### Step 1: Use the Test Set Once
The model with the best hyperparameters (already trained on the whole development set by `GridSearchCV`) is evaluated on the locked test set, once.

In [ ]:
final = search.best_estimator_
print(f"Honest test accuracy: {final.score(X_test, y_test) * 100:.1f} %   (cross-validation said {search.best_score_ * 100:.1f} %)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The test accuracy and the cross-validation score:
- Are they close? What does it tell you:

### Step 2: The Trap: Choosing on the Test Set
Now do it wrong on purpose. The cell makes 150 random forests with random hyperparameters and **picks the one with the best score on a small test set of 25 examples**. Then it asks the validation data (which the choice did not use) how good that model really is. The test score of the chosen model is optimistic.

In [ ]:
rng = np.random.default_rng(SEED)
small_X, small_y = X_rest[:300], y_rest[:300]
fresh_X, fresh_y = X_rest[300:], y_rest[300:]
best_test, best_model = -1, None
for _ in range(150):
    model = RandomForestClassifier(n_estimators=int(rng.integers(1, 15)), max_depth=int(rng.integers(1, 6)), max_features=int(rng.integers(1, 8)), random_state=int(rng.integers(0, 10000)))
    model.fit(small_X, small_y)
    score = model.score(X_test[:25], y_test[:25])               # chosen on a small "test" set of 25 examples: WRONG
    if score > best_test:
        best_test, best_model = score, model
print(f"Score of the winner on the set it was chosen on: {best_test * 100:.1f} %   <- too good")
print(f"Score of the same winner on new data:            {best_model.score(fresh_X, fresh_y) * 100:.1f} %   <- the truth")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two scores:
- Why is the first score optimistic (think: the best of 150 tries on only 25 examples):
- What should you have done to choose the model:

---
# Part G: Make It Your Own and Present (25 min)
Choose **one**:

- Draw the learning curve for the random forest of Part E and the k-nearest-neighbours model. Do they need more data?
- Tune a third hyperparameter (for example `min_samples_leaf` of the tree) with a validation curve.
- Repeat Part E with `scoring="f1_macro"`. Does the best combination change?
- Run the grid with `cv=3` and with `cv=10` and compare the time and the spread.

Then give a **3-minute demonstration**: show the spread of the score, a learning curve, a validation curve, the grid search and the trap.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- One rule of tuning that you will always follow:

> **Where next.** In Lab 40 you use all of this in your own project: you choose a problem, a baseline and a model, tune it fairly, test it once, and deliver a model with an inference demo.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The spread of the score is measured on 30 splits and with cross-validation | |
| 2 | The learning curves are drawn and one overfitting and one underfitting tree are named | |
| 3 | The validation curve for the depth and for k is drawn and the best values are chosen | |
| 4 | The grid search is run and the best combination is compared with the spread | |
| 5 | The locked test set is used once and compared with the cross-validation score | |
| 6 | The trap is shown and explained | |
| 7 | One improvement is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between a parameter and a hyperparameter? Give two examples of each.:
- 2. How can you see overfitting in the train and validation scores?:
- 3. What does a learning curve tell you that a single score does not?:
- 4. What is a validation curve? How do you choose the best value from it?:
- 5. Why does a score change when you change the random split? How does cross-validation help?:
- 6. Why must you change one hyperparameter at a time?:
- 7. Why do we keep the test set for the end?:
- 8. A friend tuned 200 models on the test set and reports the best score. What is wrong?:
- 9. A grid has 4 x 4 values and 5 folds. How many trainings?:
- 10. Your model scores 99 % on the training set and 70 % on validation. Name three things to try.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The grid search is very slow | Too many combinations or folds | Use fewer values, `cv=3`, or a smaller model; `n_jobs=-1` already uses all cores |
| `ValueError: n_splits=5 cannot be greater than the number of members in each class` | Too few examples in a class | Use `cv=3` or more data |
| The curves are very noisy | Few examples or few folds | Use more folds or repeat; remember the spread |
| The numbers are different from your classmate's | Another seed or order of cells | Run from the top; the seed is 5 |
| `ModuleNotFoundError: sklearn` | scikit-learn is not in the environment | Do Lab 19 (it installs scikit-learn) or ask your instructor |
| The pictures do not appear | The plot cell was not run | Run it again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(39)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- scikit-learn, Tuning the hyper-parameters: https://scikit-learn.org/stable/modules/grid_search.html
- scikit-learn, Learning curves and validation curves: https://scikit-learn.org/stable/modules/learning_curve.html
- Andrew Ng, Machine Learning Yearning (free book on evaluation): https://info.deeplearning.ai/machine-learning-yearning-book